# Bronze - ecommerce_enderecos

Desenvolvido por: Ygor Moraes

Este notebook lê o arquivo `ecommerce_enderecos.csv` da camada Raw e grava os dados na Bronze em Delta.

Regras aplicadas:
- preservar os dados brutos como string;
- adicionar auditoria com `bronze_ingested_at` e `bronze_source_file`;
- gerar `bronze_record_hash` para controle;
- deduplicar o lote pela chave `id_endereco`;
- particionar por `ano` e `mes` com base na data de ingestão;
- gravar em modo overwrite para recriar a Bronze com a carga full atual.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define imports, caminhos e parâmetros da Bronze de endereços.

from pyspark.sql.functions import (
    col,
    count,
    when,
    current_timestamp,
    year,
    month,
    sha2,
    concat_ws,
    coalesce,
    lit
)

SOURCE_FILE = "ecommerce_enderecos.csv"
SOURCE_PATH = f"{RAW_BATCH_PATH}{SOURCE_FILE}"

BRONZE_TABLE = "ecommerce_enderecos"
BRONZE_PATH = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"

CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "false"
}

EXPECTED_COLUMNS = [
    "id_endereco",
    "id_cliente",
    "logradouro",
    "numero",
    "complemento",
    "bairro",
    "cep",
    "cidade",
    "estado",
    "latitude",
    "longitude",
    "apelido",
    "is_principal"
]

KEY_COLUMNS = ["id_endereco"]
DEDUP_COLUMNS = ["id_endereco"]

BRONZE_WRITE_MODE = "overwrite"

adls_options = get_adls_options()

print("Notebook configurado.")
print(f"Origem Raw: {SOURCE_PATH}")
print(f"Destino Bronze: {BRONZE_PATH}")
print(f"Modo de escrita: {BRONZE_WRITE_MODE}")

In [0]:
# Lê o CSV da Raw e valida se as colunas esperadas existem.

df_source = read_source_csv(
    spark=spark,
    source_path=SOURCE_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS
)

actual_columns = df_source.columns

missing_columns = [c for c in EXPECTED_COLUMNS if c not in actual_columns]
extra_columns = [c for c in actual_columns if c not in EXPECTED_COLUMNS]

if missing_columns:
    raise Exception(f"Colunas obrigatórias ausentes na origem: {missing_columns}")

if extra_columns:
    print(f"Atenção: colunas extras encontradas na origem: {extra_columns}")

total_source = df_source.count()

print("Validação inicial OK.")
print(f"Total de registros lidos da Raw: {total_source}")
print(f"Total de colunas: {len(actual_columns)}")

df_source.printSchema()

display(df_source.limit(10))

In [0]:
# Valida a chave principal da tabela de endereços.

df_validacao_chave = df_source.select(
    count("*").alias("total_linhas"),
    count(when(col("id_endereco").isNull(), True)).alias("id_endereco_nulo")
)

display(df_validacao_chave)

total_distintos = df_source.select("id_endereco").distinct().count()
duplicados = total_source - total_distintos

print(f"Total de linhas: {total_source}")
print(f"IDs distintos: {total_distintos}")
print(f"IDs duplicados: {duplicados}")

if duplicados > 0:
    print("Atenção: existem id_endereco duplicados. Eles serão tratados antes da escrita.")

print("Validação de chave concluída.")

In [0]:
# Prepara a Bronze com auditoria, hash e partições por data de ingestão.

df_bronze = (
    df_source
    .select(*EXPECTED_COLUMNS)
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("bronze_source_file", lit(SOURCE_FILE))
    .withColumn(
        "bronze_record_hash",
        sha2(
            concat_ws(
                "||",
                *[coalesce(col(c), lit("")) for c in EXPECTED_COLUMNS]
            ),
            256
        )
    )
    .withColumn("ano", year(col("bronze_ingested_at")))
    .withColumn("mes", month(col("bronze_ingested_at")))
)

df_bronze_batch = df_bronze.dropDuplicates(DEDUP_COLUMNS)

print("DataFrame Bronze preparado.")
print(f"Total antes da deduplicação: {df_bronze.count()}")
print(f"Total após deduplicação: {df_bronze_batch.count()}")

display(df_bronze_batch.limit(10))

In [0]:
# Grava a Bronze de endereços em Delta com overwrite.

total_to_write = df_bronze_batch.count()

(
    df_bronze_batch
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(BRONZE_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(BRONZE_PATH)
)

print(f"Bronze recriada com sucesso em Delta: {BRONZE_PATH}")
print(f"Modo de escrita utilizado: {BRONZE_WRITE_MODE}")
print(f"Total gravado na Bronze: {total_to_write}")

In [0]:
# Valida volume, duplicidade, partições e campos de auditoria após a escrita.

df_bronze_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(BRONZE_PATH)
)

total_bronze_saved = df_bronze_saved.count()

print(f"Total preparado para gravação: {total_to_write}")
print(f"Total Bronze gravada: {total_bronze_saved}")

if total_bronze_saved != total_to_write:
    raise Exception("Erro: total da Bronze gravada não bate com o total preparado.")

df_validacao_bronze = df_bronze_saved.select(
    count("*").alias("total_linhas"),
    count(when(col("id_endereco").isNull(), True)).alias("id_endereco_nulo"),
    count(when(col("ano").isNull(), True)).alias("ano_nulo"),
    count(when(col("mes").isNull(), True)).alias("mes_nulo"),
    count(when(col("bronze_ingested_at").isNull(), True)).alias("bronze_ingested_at_nulo"),
    count(when(col("bronze_source_file").isNull(), True)).alias("bronze_source_file_nulo"),
    count(when(col("bronze_record_hash").isNull(), True)).alias("bronze_record_hash_nulo")
)

display(df_validacao_bronze)

validacao_bronze = df_validacao_bronze.collect()[0]

duplicados_bronze = (
    total_bronze_saved
    - df_bronze_saved.select("id_endereco").distinct().count()
)

print(f"IDs duplicados na Bronze: {duplicados_bronze}")

if validacao_bronze["id_endereco_nulo"] > 0:
    raise Exception("Erro: existem registros com id_endereco nulo na Bronze.")

if validacao_bronze["ano_nulo"] > 0:
    raise Exception("Erro: existem registros com ano nulo na Bronze.")

if validacao_bronze["mes_nulo"] > 0:
    raise Exception("Erro: existem registros com mes nulo na Bronze.")

if validacao_bronze["bronze_ingested_at_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_ingested_at.")

if validacao_bronze["bronze_source_file_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_source_file.")

if validacao_bronze["bronze_record_hash_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_record_hash.")

if duplicados_bronze > 0:
    raise Exception("Erro: existem id_endereco duplicados na Bronze.")

print("Validação final da Bronze full OK.")